In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CODE_DIR = '/content/drive/MyDrive/CSE720/code'
os.makedirs(CODE_DIR, exist_ok=True)
print('Code will be saved to:', CODE_DIR)

Mounted at /content/drive
Code will be saved to: /content/drive/MyDrive/CSE720/code


In [ ]:
!pip install -q torch torchvision torchaudio scikit-image scipy tqdm pandas matplotlib seaborn statsmodels

In [ ]:
CODE_DIR = '/content/drive/MyDrive/CSE720/code'

config_py = r'''
import torch
import os

class Config:
    img_size = 64
    batch_size = 8
    channels = 3
    num_domains = 5

    lr = 0.001
    beta1 = 0.5
    beta2 = 0.999
    num_epochs = 100

    # Loss weights (unchanged from the original EyeGAN run)
    lambda_perceptual = 10.0
    lambda_gp = 10.0
    lambda_cycle = 5.0
    lambda_cls = 1.0
    lambda_identity = 5.0

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    dataset_path = '/content/drive/MyDrive/CSE720/EyeGAN'
    base_dir = '/content/drive/MyDrive/CSE720'

    # Original (full-model) checkpoint, produced by StarGan_EyeGan.ipynb
    checkpoint_dir = '/content/drive/MyDrive/CSE720/stargan_checkpoints'
    sample_dir = '/content/drive/MyDrive/CSE720/Samples'

    # New: root for everything the revision notebooks produce, kept
    # separate from the original run so nothing gets overwritten.
    revision_dir = '/content/drive/MyDrive/CSE720/revision'
    ablation_dir = os.path.join(revision_dir, 'ablation')
    eval_dir = os.path.join(revision_dir, 'evaluation')
    classifier_dir = os.path.join(revision_dir, 'downstream_classifier')
    benchmark_dir = os.path.join(revision_dir, 'benchmark')
    external_dir = os.path.join(revision_dir, 'external_validation')
    expert_dir = os.path.join(revision_dir, 'expert_review')

    for d in [revision_dir, ablation_dir, eval_dir, classifier_dir,
              benchmark_dir, external_dir, expert_dir]:
        os.makedirs(d, exist_ok=True)

    eval_freq = 5
    save_freq = 10

    class_names = {
        0: 'Diabetic_Retinopathy',
        1: 'Glaucoma',
        2: 'Healthy',
        3: 'Macular_Scar',
        4: 'Myopia'
    }
    class_colors = {0: 'red', 1: 'blue', 2: 'green', 3: 'orange', 4: 'purple'}
    use_amp = False


class AblationVariant:
    """
    One entry per ablation arm requested by Reviewer #1 (Q5) and
    Reviewer #2 (Q3c): each turns OFF exactly one loss term relative
    to the full EyeGAN objective. Everything else (architecture,
    epochs, optimizer, data) is held fixed so the comparison isolates
    that one term.
    """
    VARIANTS = {
        'full_model':      dict(use_cls=True,  use_cycle=True,  use_identity=True,  use_gp=True,  use_perceptual=True),
        'no_cls':          dict(use_cls=False, use_cycle=True,  use_identity=True,  use_gp=True,  use_perceptual=True),
        'no_cycle':        dict(use_cls=True,  use_cycle=False, use_identity=True,  use_gp=True,  use_perceptual=True),
        'no_identity':     dict(use_cls=True,  use_cycle=True,  use_identity=False, use_gp=True,  use_perceptual=True),
        'no_gp':           dict(use_cls=True,  use_cycle=True,  use_identity=True,  use_gp=False, use_perceptual=True),
    }

    @staticmethod
    def get(name):
        assert name in AblationVariant.VARIANTS, f"Unknown variant '{name}'. Choose from {list(AblationVariant.VARIANTS)}"
        return AblationVariant.VARIANTS[name]
'''
with open(f'{CODE_DIR}/config.py', 'w') as f:
    f.write(config_py)
print('config.py written')

config.py written


In [ ]:
dataset_py = r'''
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from config import Config

class MedicalDataset(Dataset):
    def __init__(self, root_dir, image_size=64, mode='train', train_ratio=0.8, val_ratio=0.1):
        self.root_dir = root_dir
        self.image_size = image_size
        self.mode = mode
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.cfg = Config()

        self.domains = sorted([d.strip() for d in os.listdir(root_dir)
                             if os.path.isdir(os.path.join(root_dir, d))])

        self.image_paths = []
        self.labels = []
        self.domain_to_label = {domain: idx for idx, domain in enumerate(self.domains)}
        self.label_to_domain = {idx: domain for domain, idx in self.domain_to_label.items()}

        domain_stats = {}
        for domain in self.domains:
            domain_path = os.path.join(root_dir, domain)
            if not os.path.exists(domain_path):
                continue
            domain_images = sorted([os.path.join(domain_path, img) for img in os.listdir(domain_path)
                           if img.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
            if len(domain_images) == 0:
                continue

            total_images = len(domain_images)
            train_split = int(total_images * train_ratio)
            val_split = int(total_images * (train_ratio + val_ratio))

            if mode == 'train':
                selected_images = domain_images[:train_split]
            elif mode == 'val':
                selected_images = domain_images[train_split:val_split]
            elif mode == 'test':
                selected_images = domain_images[val_split:]
            elif mode == 'all':
                # Every original (non-augmented) image in the domain,
                # regardless of split. Used ONLY as the real-image
                # reference distribution for FID (see metrics.py) -
                # never mixed into train/val/test.
                selected_images = domain_images
            else:
                raise ValueError(f"Unknown mode {mode}")

            label = self.domain_to_label[domain]
            self.image_paths.extend(selected_images)
            self.labels.extend([label] * len(selected_images))
            domain_stats[domain] = len(selected_images)

        print(f"[{mode}] domains found: {domain_stats}  (total {len(self.image_paths)})")

        if mode == 'train':
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(p=0.3),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
            ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)
            label = self.labels[idx]
            return image, label, img_path
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            placeholder = torch.zeros(3, self.image_size, self.image_size)
            return placeholder, self.labels[idx], "error_path"

    def get_class_weights(self):
        from collections import Counter
        label_counts = Counter(self.labels)
        total_samples = len(self.labels)
        return {label: total_samples / count for label, count in label_counts.items()}
'''
with open(f'{CODE_DIR}/dataset.py', 'w') as f:
    f.write(dataset_py)
print('dataset.py written')

dataset.py written


In [ ]:
model_py = r'''
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1),
            nn.InstanceNorm2d(dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim, dim, 3, 1, 1),
            nn.InstanceNorm2d(dim)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, img_size=64, conv_dim=32, num_domains=5):
        super().__init__()
        self.img_size = img_size
        self.num_domains = num_domains

        self.label_emb = nn.Sequential(
            nn.Embedding(num_domains, 128),
            nn.Linear(128, img_size * img_size),
            nn.ReLU()
        )

        self.initial_conv = nn.Sequential(
            nn.Conv2d(4, conv_dim, 7, 1, 3),
            nn.InstanceNorm2d(conv_dim),
            nn.ReLU(inplace=True)
        )
        self.downsample1 = nn.Sequential(
            nn.Conv2d(conv_dim, conv_dim*2, 4, 2, 1),
            nn.InstanceNorm2d(conv_dim*2),
            nn.ReLU(inplace=True)
        )
        self.downsample2 = nn.Sequential(
            nn.Conv2d(conv_dim*2, conv_dim*4, 4, 2, 1),
            nn.InstanceNorm2d(conv_dim*4),
            nn.ReLU(inplace=True)
        )
        self.res_blocks = nn.Sequential(*[ResidualBlock(conv_dim*4) for _ in range(4)])
        self.upsample1 = nn.Sequential(
            nn.ConvTranspose2d(conv_dim*4, conv_dim*2, 4, 2, 1),
            nn.InstanceNorm2d(conv_dim*2),
            nn.ReLU(inplace=True)
        )
        self.upsample2 = nn.Sequential(
            nn.ConvTranspose2d(conv_dim*2, conv_dim, 4, 2, 1),
            nn.InstanceNorm2d(conv_dim),
            nn.ReLU(inplace=True)
        )
        self.final_conv = nn.Sequential(
            nn.Conv2d(conv_dim, 3, 7, 1, 3),
            nn.Tanh()
        )

    def forward(self, x, domain_label):
        batch_size = x.size(0)
        label_emb = self.label_emb(domain_label).view(batch_size, 1, self.img_size, self.img_size)
        x = torch.cat([x, label_emb], dim=1)
        x = self.initial_conv(x)
        x = self.downsample1(x)
        x = self.downsample2(x)
        x = self.res_blocks(x)
        x = self.upsample1(x)
        x = self.upsample2(x)
        x = self.final_conv(x)
        return x

class Discriminator(nn.Module):
    def __init__(self, img_size=64, conv_dim=32, num_domains=5):
        super().__init__()
        self.img_size = img_size
        self.num_domains = num_domains
        layers = [
            nn.Conv2d(3, conv_dim, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(conv_dim, conv_dim*2, 4, 2, 1),
            nn.InstanceNorm2d(conv_dim*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(conv_dim*2, conv_dim*4, 4, 2, 1),
            nn.InstanceNorm2d(conv_dim*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(conv_dim*4, conv_dim*8, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
        ]
        self.feature_extractor = nn.Sequential(*layers)
        self.real_fake = nn.Conv2d(conv_dim*8, 1, 3, 1, 1)
        self.domain_cls = nn.Sequential(
            nn.Conv2d(conv_dim*8, num_domains, 3, 1, 1),
            nn.AdaptiveAvgPool2d(1)
        )

    def forward(self, x):
        features = self.feature_extractor(x)
        out_src = self.real_fake(features)
        out_cls = self.domain_cls(features)
        return out_src.squeeze(), out_cls.squeeze()
'''
with open(f'{CODE_DIR}/model.py', 'w') as f:
    f.write(model_py)
print('model.py written')

model.py written


In [ ]:
losses_py = r'''
import torch
import torch.nn.functional as F
import torchvision.models as models

class LossCalculator:
    def __init__(self, device, cfg):
        self.device = device
        self.cfg = cfg
        self.vgg = self._init_vgg().to(device).eval()

    def _init_vgg(self):
        vgg = models.vgg16(weights='IMAGENET1K_V1').features[:10]
        for p in vgg.parameters():
            p.requires_grad = False
        return vgg

    def adversarial_loss(self, logits, target_is_real):
        target = torch.ones_like(logits) if target_is_real else torch.zeros_like(logits)
        return F.mse_loss(logits, target)

    def classification_loss(self, pred_cls, target_cls):
        return F.cross_entropy(pred_cls, target_cls)

    def reconstruction_loss(self, x, x_recon):
        return F.l1_loss(x, x_recon)

    def perceptual_loss(self, real_img, fake_img):
        return F.l1_loss(self.vgg(real_img), self.vgg(fake_img))

    def identity_loss(self, input_img, same_domain_output):
        return F.l1_loss(input_img, same_domain_output)

    def gradient_penalty(self, D, real_imgs, fake_imgs):
        batch_size = real_imgs.size(0)
        alpha = torch.rand(batch_size, 1, 1, 1).to(self.device)
        interpolates = (alpha * real_imgs + (1 - alpha) * fake_imgs).requires_grad_(True)
        d_interpolates, _ = D(interpolates)
        gradients = torch.autograd.grad(
            outputs=d_interpolates, inputs=interpolates,
            grad_outputs=torch.ones_like(d_interpolates),
            create_graph=True, retain_graph=True)[0]
        gradients = gradients.view(batch_size, -1)
        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

def create_label_tensor(label, batch_size, num_domains):
    target = torch.randint(0, num_domains, (batch_size,))
    for i in range(batch_size):
        while target[i] == label[i]:
            target[i] = torch.randint(0, num_domains, (1,))
    return target

def create_same_label_tensor(label, batch_size):
    return label.clone()

def denormalize_image(tensor):
    return tensor * 0.5 + 0.5
'''
with open(f'{CODE_DIR}/losses.py', 'w') as f:
    f.write(losses_py)
print('losses.py written')

losses.py written


In [ ]:
metrics_py = r'''
import os
import csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from scipy import linalg
from skimage.metrics import structural_similarity as ssim_fn
from torch.utils.data import DataLoader
from dataset import MedicalDataset

class FIDCalculator:
    def __init__(self, device):
        self.device = device
        # Use default pretrained weights to avoid aux_logits error in PyTorch 2.x+
        weights = models.Inception_V3_Weights.DEFAULT
        inception = models.inception_v3(weights=weights)

        # Replace the fully connected layer to output 2048-dim feature vector directly
        inception.fc = nn.Identity()
        self.inception = inception.to(device)
        self.inception.eval()

        for p in self.inception.parameters():
            p.requires_grad = False

    def _prep(self, images):
        # Scale images from [-1, 1] to [0, 1]
        images = images * 0.5 + 0.5
        images = torch.clamp(images, 0, 1)
        # Resize images to 299x299 for Inception V3
        if images.shape[2] != 299 or images.shape[3] != 299:
            images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
        return images

    @torch.no_grad()
    def get_activations(self, images, batch_size=32):
        acts = []
        for i in range(0, images.size(0), batch_size):
            batch = self._prep(images[i:i+batch_size]).to(self.device)
            out = self.inception(batch)

            # Handle InceptionOutputs if present
            if hasattr(out, 'logits'):
                out = out.logits

            acts.append(out.cpu().numpy())
        return np.concatenate(acts, axis=0)

    def calculate_fid(self, real_acts, fake_acts):
        mu1, sigma1 = real_acts.mean(axis=0), np.cov(real_acts, rowvar=False)
        mu2, sigma2 = fake_acts.mean(axis=0), np.cov(fake_acts, rowvar=False)
        diff = mu1 - mu2
        covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
        if np.iscomplexobj(covmean):
            covmean = covmean.real
        return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))


def calc_psnr_ssim_mse(real_img, fake_img):
    real = (real_img * 0.5 + 0.5).clamp(0, 1)
    fake = (fake_img * 0.5 + 0.5).clamp(0, 1)
    mse = torch.mean((real - fake) ** 2).item()
    psnr = 20 * np.log10(1.0 / np.sqrt(mse)) if mse > 0 else float('inf')
    real_np = real.cpu().numpy().transpose(1, 2, 0)
    fake_np = fake.cpu().numpy().transpose(1, 2, 0)
    ssim_val = ssim_fn(real_np, fake_np, channel_axis=2, data_range=1.0)
    return psnr, ssim_val, mse


@torch.no_grad()
def evaluate_full_test_set(G, cfg, device, out_dir, model_name='EyeGAN', max_images=None):
    """
    Full-test-set evaluation:
      - Every image in the TEST split is translated to every OTHER domain
      - PSNR / SSIM / MSE are computed per (image, target) pair and saved as a CSV
      - FID is computed PER TARGET DOMAIN using the full pool as reference
    """
    os.makedirs(out_dir, exist_ok=True)
    test_dataset = MedicalDataset(cfg.dataset_path, cfg.img_size, 'test')
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    fid_calc = FIDCalculator(device)
    G.eval()

    per_item_rows = []
    fake_by_domain = {d: [] for d in range(cfg.num_domains)}

    n_seen = 0
    for real_img, label, path in test_loader:
        if max_images is not None and n_seen >= max_images:
            break
        real_img = real_img.to(device)
        source_label = label.item()
        img_name = os.path.basename(path[0])

        for target_domain in range(cfg.num_domains):
            target_tensor = torch.tensor([target_domain], device=device)
            fake_img = G(real_img, target_tensor)[0]
            fake_by_domain[target_domain].append(fake_img.detach().cpu())

            if target_domain != source_label:
                psnr_v, ssim_v, mse_v = calc_psnr_ssim_mse(real_img[0], fake_img)
                per_item_rows.append({
                    'model': model_name, 'image': img_name,
                    'source': cfg.class_names[source_label],
                    'target': cfg.class_names[target_domain],
                    'psnr': psnr_v, 'ssim': ssim_v, 'mse': mse_v
                })
        n_seen += 1

    # Per-domain FID calculation
    fid_rows = []
    for target_domain in range(cfg.num_domains):
        fake_imgs = torch.stack(fake_by_domain[target_domain])
        real_pool = MedicalDataset(cfg.dataset_path, cfg.img_size, 'all')

        idxs = [i for i, l in enumerate(real_pool.labels) if l == target_domain]
        real_imgs = torch.stack([real_pool[i][0] for i in idxs])

        real_acts = fid_calc.get_activations(real_imgs)
        fake_acts = fid_calc.get_activations(fake_imgs)
        fid_val = fid_calc.calculate_fid(real_acts, fake_acts)

        fid_rows.append({
            'model': model_name,
            'target': cfg.class_names[target_domain],
            'fid': fid_val,
            'n_real': len(real_imgs),
            'n_fake': len(fake_imgs)
        })

    # Save CSV files
    item_csv = os.path.join(out_dir, f'{model_name}_per_item_results.csv')
    with open(item_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'image', 'source', 'target', 'psnr', 'ssim', 'mse'])
        w.writeheader()
        w.writerows(per_item_rows)

    fid_csv = os.path.join(out_dir, f'{model_name}_fid_per_domain.csv')
    with open(fid_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'target', 'fid', 'n_real', 'n_fake'])
        w.writeheader()
        w.writerows(fid_rows)

    psnr_vals = [r['psnr'] for r in per_item_rows if np.isfinite(r['psnr'])]
    ssim_vals = [r['ssim'] for r in per_item_rows]
    mse_vals = [r['mse'] for r in per_item_rows]
    fid_vals = [r['fid'] for r in fid_rows]

    summary = {
        'model': model_name,
        'n_translations': len(per_item_rows),
        'n_source_images': n_seen,
        'psnr_mean': float(np.mean(psnr_vals)), 'psnr_std': float(np.std(psnr_vals)),
        'ssim_mean': float(np.mean(ssim_vals)), 'ssim_std': float(np.std(ssim_vals)),
        'mse_mean': float(np.mean(mse_vals)), 'mse_std': float(np.std(mse_vals)),
        'fid_mean': float(np.mean(fid_vals)), 'fid_std': float(np.std(fid_vals)),
    }
    print(f"[{model_name}] n_translations={summary['n_translations']} (should be {n_seen}x4= {n_seen*4})")
    print(f"[{model_name}] PSNR {summary['psnr_mean']:.2f}±{summary['psnr_std']:.2f} | "
          f"SSIM {summary['ssim_mean']:.3f}±{summary['ssim_std']:.3f} | "
          f"MSE {summary['mse_mean']:.5f}±{summary['mse_std']:.5f} | "
          f"FID {summary['fid_mean']:.2f}±{summary['fid_std']:.2f}")

    return per_item_rows, summary, fid_rows
'''

with open(f'{CODE_DIR}/metrics.py', 'w') as f:
    f.write(metrics_py)
print('metrics.py written successfully')

metrics.py written successfully


In [ ]:
train_lib_py = r'''
import os
import time
import torch
import gc
from torch.utils.data import DataLoader
from dataset import MedicalDataset
from model import Generator, Discriminator
from losses import LossCalculator, create_label_tensor, create_same_label_tensor, denormalize_image
from config import Config, AblationVariant
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import json

def train_variant(variant_name, num_epochs=None, resume=True):
    """
    Trains one ablation arm. Safe to stop/restart across Colab sessions:
    if a checkpoint for this variant already exists it resumes from the
    latest epoch instead of starting over (important since free-tier
    Colab sessions get cut off).
    """
    flags = AblationVariant.get(variant_name)
    cfg = Config()
    device = cfg.device
    epochs_total = num_epochs or cfg.num_epochs

    variant_dir = os.path.join(cfg.ablation_dir, variant_name)
    ckpt_dir = os.path.join(variant_dir, 'checkpoints')
    sample_dir = os.path.join(variant_dir, 'samples')
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(sample_dir, exist_ok=True)

    print(f"=== Training variant: {variant_name} ===")
    print("Loss flags:", flags)

    torch.cuda.empty_cache(); gc.collect()

    train_dataset = MedicalDataset(cfg.dataset_path, cfg.img_size, 'train')
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size,
                               shuffle=True, drop_last=True, num_workers=0, pin_memory=True)

    G = Generator(cfg.img_size, num_domains=cfg.num_domains).to(device)
    D = Discriminator(cfg.img_size, num_domains=cfg.num_domains).to(device)
    g_optimizer = torch.optim.Adam(G.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    d_optimizer = torch.optim.Adam(D.parameters(), lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))
    g_scheduler = torch.optim.lr_scheduler.StepLR(g_optimizer, step_size=20, gamma=0.5)
    d_scheduler = torch.optim.lr_scheduler.StepLR(d_optimizer, step_size=20, gamma=0.5)
    loss_calc = LossCalculator(device, cfg)

    start_epoch = 0
    history = {'d_loss': [], 'g_loss': [], 'epoch_time_sec': []}
    latest_ckpt = os.path.join(ckpt_dir, 'latest.pth')
    if resume and os.path.exists(latest_ckpt):
        ck = torch.load(latest_ckpt, map_location=device, weights_only=False)
        G.load_state_dict(ck['G_state_dict']); D.load_state_dict(ck['D_state_dict'])
        g_optimizer.load_state_dict(ck['g_optimizer_state_dict'])
        d_optimizer.load_state_dict(ck['d_optimizer_state_dict'])
        g_scheduler.load_state_dict(ck['g_scheduler_state_dict'])
        d_scheduler.load_state_dict(ck['d_scheduler_state_dict'])
        history = ck['history']
        start_epoch = ck['epoch']
        print(f"Resumed from epoch {start_epoch}")

    if start_epoch >= epochs_total:
        print(f"Variant '{variant_name}' already trained for {start_epoch} epochs (target {epochs_total}). Nothing to do.")
        return

    for epoch in range(start_epoch, epochs_total):
        G.train(); D.train()
        t0 = time.time()
        loop = tqdm(train_loader, desc=f"[{variant_name}] Epoch [{epoch+1}/{epochs_total}]")

        for imgs, labels, _ in loop:
            imgs = imgs.to(device); labels = labels.to(device)
            batch_size = imgs.size(0)
            target_labels = create_label_tensor(labels, batch_size, cfg.num_domains).to(device)
            same_labels = create_same_label_tensor(labels, batch_size).to(device)

            # --- Discriminator ---
            d_optimizer.zero_grad()
            real_out_src, real_out_cls = D(imgs)
            d_loss_real = loss_calc.adversarial_loss(real_out_src, True)
            d_loss_cls = loss_calc.classification_loss(real_out_cls, labels)
            fake_imgs = G(imgs, target_labels)
            fake_out_src, _ = D(fake_imgs.detach())
            d_loss_fake = loss_calc.adversarial_loss(fake_out_src, False)

            d_loss = d_loss_real + d_loss_fake
            if flags['use_cls']:
                d_loss = d_loss + cfg.lambda_cls * d_loss_cls
            if flags['use_gp']:
                d_loss_gp = loss_calc.gradient_penalty(D, imgs, fake_imgs.detach())
                d_loss = d_loss + cfg.lambda_gp * d_loss_gp
            d_loss.backward()
            d_optimizer.step()

            # --- Generator ---
            g_optimizer.zero_grad()
            fake_out_src, fake_out_cls = D(fake_imgs)
            g_loss_adv = loss_calc.adversarial_loss(fake_out_src, True)
            g_loss_cls = loss_calc.classification_loss(fake_out_cls, target_labels)
            g_loss = g_loss_adv
            if flags['use_cls']:
                g_loss = g_loss + cfg.lambda_cls * g_loss_cls
            if flags['use_cycle']:
                rec_imgs = G(fake_imgs, labels)
                g_loss_rec = loss_calc.reconstruction_loss(imgs, rec_imgs)
                g_loss = g_loss + cfg.lambda_cycle * g_loss_rec
            if flags['use_identity']:
                identity_imgs = G(imgs, same_labels)
                g_loss_id = loss_calc.identity_loss(imgs, identity_imgs)
                g_loss = g_loss + cfg.lambda_identity * g_loss_id
            if flags['use_perceptual']:
                g_loss_perc = loss_calc.perceptual_loss(imgs, fake_imgs)
                g_loss = g_loss + cfg.lambda_perceptual * g_loss_perc

            g_loss.backward()
            g_optimizer.step()

            loop.set_postfix(D=f"{d_loss.item():.3f}", G=f"{g_loss.item():.3f}")
            history['d_loss'].append(d_loss.item())
            history['g_loss'].append(g_loss.item())

        g_scheduler.step(); d_scheduler.step()
        history['epoch_time_sec'].append(time.time() - t0)

        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                G.eval()
                sample_imgs, sample_labels, _ = next(iter(train_loader))
                sample_imgs = sample_imgs[:4].to(device)
                fig, axes = plt.subplots(1, 4, figsize=(12, 3))
                for i in range(4):
                    gen = G(sample_imgs[i:i+1], torch.tensor([0], device=device))[0]
                    img_np = np.clip((gen.cpu().detach() * 0.5 + 0.5).numpy().transpose(1, 2, 0), 0, 1)
                    axes[i].imshow(img_np); axes[i].axis('off')
                plt.suptitle(f'{variant_name} epoch {epoch+1}')
                plt.savefig(os.path.join(sample_dir, f'epoch_{epoch+1}.png'), dpi=100, bbox_inches='tight')
                plt.close()
                G.train()

        # Save "latest" every epoch (cheap, ~model-size only) so a
        # disconnect never loses more than one epoch of work.
        ckpt = {
            'epoch': epoch + 1, 'variant': variant_name, 'flags': flags,
            'G_state_dict': G.state_dict(), 'D_state_dict': D.state_dict(),
            'g_optimizer_state_dict': g_optimizer.state_dict(),
            'd_optimizer_state_dict': d_optimizer.state_dict(),
            'g_scheduler_state_dict': g_scheduler.state_dict(),
            'd_scheduler_state_dict': d_scheduler.state_dict(),
            'history': history
        }
        torch.save(ckpt, latest_ckpt)
        if (epoch + 1) % cfg.save_freq == 0:
            torch.save(ckpt, os.path.join(ckpt_dir, f'checkpoint_epoch_{epoch+1}.pth'))

    with open(os.path.join(variant_dir, 'training_log.json'), 'w') as f:
        json.dump({'variant': variant_name, 'flags': flags,
                    'total_epochs': epochs_total,
                    'mean_epoch_time_sec': float(np.mean(history['epoch_time_sec'])),
                    'final_g_loss': history['g_loss'][-1] if history['g_loss'] else None,
                    'final_d_loss': history['d_loss'][-1] if history['d_loss'] else None}, f, indent=2)

    print(f"=== Finished '{variant_name}': {epochs_total} epochs, "
          f"mean epoch time {np.mean(history['epoch_time_sec']):.1f}s ===")
    return G, D, history
'''
with open(f'{CODE_DIR}/train_lib.py', 'w') as f:
    f.write(train_lib_py)
print('train_lib.py written')

train_lib.py written


In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSE720/code')
from config import Config
from dataset import MedicalDataset

cfg = Config()
print("Dataset source path:", cfg.dataset_path)

rows = []
for mode in ['train', 'val', 'test']:
    ds = MedicalDataset(cfg.dataset_path, cfg.img_size, mode)
    from collections import Counter
    counts = Counter(ds.labels)
    for label_idx, count in sorted(counts.items()):
        rows.append({'split': mode, 'class': cfg.class_names[label_idx], 'n_images': count})

import pandas as pd
df = pd.DataFrame(rows)
pivot = df.pivot(index='class', columns='split', values='n_images').fillna(0).astype(int)
pivot['total'] = pivot.sum(axis=1)
print(pivot)
pivot.to_csv('/content/drive/MyDrive/CSE720/revision/dataset_statistics.csv')
print("\\nSaved to revision/dataset_statistics.csv — paste this table into the Response to Reviewers"
      " for R1-Q1 (exact images per class / split).")

Dataset source path: /content/drive/MyDrive/CSE720/EyeGAN
[train] domains found: {'Diabetic Retinopathy': 400, 'Glaucoma': 400, 'Healthy': 400, 'Macular Scar': 400, 'Myopia': 400}  (total 2000)
[val] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
[test] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
split                 test  train  val  total
class                                        
Diabetic_Retinopathy    50    400   50    500
Glaucoma                50    400   50    500
Healthy                 50    400   50    500
Macular_Scar            50    400   50    500
Myopia                  50    400   50    500
\nSaved to revision/dataset_statistics.csv — paste this table into the Response to Reviewers for R1-Q1 (exact images per class / split).
